# Réplication et interprétation experte : VECM sur les données macroéconomiques canadiennes

**Référence principale** : Bernhard Pfaff (2008), *VAR, SVAR and SVEC Models: Implementation Within R Package vars*, *Journal of Statistical Software*, 27(4).  
**Données** : Canada (OECD / JMULTI), 84 observations trimestrielles (1980Q1–2000Q4).  
**Variables** :  
- `e` : emploi (100 × ln(emploi civil))  
- `prod` : productivité du travail (mesure construite)  
- `rw` : salaire réel  
- `U` : taux de chômage  

**Objectif** : Reproduire exactement les résultats de l’article JSS (sélection d’ordre, test de Johansen case 4, estimation VECM r=1, vecteur de cointégration normalisé et coefficients d’ajustement), puis fournir une interprétation économétrique et économique rigoureuse. Comparaison complémentaire avec `statsmodels`.

**Packages utilisés** : `makystats` ≥ 0.1.1 (VAR natif + VECM / Johansen) et `statsmodels` (comparaison).

## 0. Imports et chargement des données

In [2]:
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.vecm import VECM as SM_VECM
from makystats import VAR, select_var_order, VECM, select_coint_rank
from makystats.vecm import summarize_johansen

pd.set_option("display.precision", 4)
pd.set_option("display.width", 120)

# Chargement
can = pd.read_csv("canada.csv", index_col=0)
can.index = pd.PeriodIndex(can.index, freq="Q")
print("Shape :", can.shape)
print("Colonnes :", can.columns.tolist())
print("\nPériode :", can.index[0], "→", can.index[-1])
can.head()

Shape : (84, 4)
Colonnes : ['e', 'prod', 'rw', 'U']

Période : 1980Q1 → 2000Q4


,e,prod,rw,U
1980Q1,929.6105,405.3665,386.1361,7.53
1980Q2,929.8040,404.6398,388.1358,7.70
1980Q3,930.3184,403.8149,390.5401,7.47
1980Q4,931.4277,404.2158,393.9638,7.27
1981Q1,932.6620,405.0467,396.7647,7.37


### Commentaire sur les données

Les quatre séries sont classiquement considérées comme I(1). La productivité (`prod`) et l’emploi (`e`) présentent une tendance stochastique commune avec le salaire réel (`rw`), tandis que le chômage (`U`) est une variable de slack du marché du travail.  
L’ordre des variables n’est pas anodin pour la normalisation du vecteur de cointégration : dans l’article JSS (et dans `ca.jo` de `urca`), on utilise l’ordre `rw, prod, e, U` afin de normaliser sur le salaire réel (coefficient de `rw` = 1). Nous respecterons strictement cet ordre lors de l’estimation VECM.

## 1. Sélection de l’ordre de retards (VAR en niveaux)

On estime un VAR en niveaux avec constante **et** tendance linéaire (`det="ct"`), comme dans l’article.  
La sélection d’ordre est réalisée avec **`makystats.select_var_order`** (échantillon commun, critères AIC / BIC / HQIC).

In [3]:
order = select_var_order(can, maxlags=8, det="ct", criterion="aic", verbose=True)
print("\nOrdres retenus :", order.selected)
print("Meilleur selon AIC :", order.best)
print("Article JSS       : AIC=3, HQ=2, SC=1, FPE=3")

VAR lag order selection (det='ct', criterion='aic')
Selected: AIC=3, BIC=1, HQIC=2  → best=3
            llf        aic        bic       hqic  nobs  n_params    ok
k_ar                                                                  
3     -118.0526     4.5803     6.2977     5.2667    76        56  True
2     -139.1639     4.7148     5.9415     5.2051    76        40  True
4     -107.2422     4.7169     6.9250     5.5993    76        72  True
5     -100.2283     4.9534     7.6521     6.0319    76        88  True
6      -87.6645     5.0438     8.2332     6.3185    76       104  True
1     -168.9993     5.0789     5.8150     5.3731    76        24  True
7      -80.6697     5.2808     8.9609     6.7515    76       120  True
8      -65.0166     5.2899     9.4607     6.9568    76       136  True

Ordres retenus : {'aic': 3, 'bic': 1, 'hqic': 2}
Meilleur selon AIC : 3
Article JSS       : AIC=3, HQ=2, SC=1, FPE=3


### Interprétation experte

- **AIC** sélectionne **p = 3**.  
- **HQIC** sélectionne **p = 2**.  
- **BIC (SC)** sélectionne **p = 1** (plus parcimonieux).

Dans le contexte de la cointégration de Johansen, on privilégie généralement un ordre légèrement plus élevé (AIC/HQ) pour éviter un biais de sous-spécification des dynamiques de court terme qui contaminerait le rang de cointégration. L’article JSS retient donc les candidats p = 2 et p = 3 pour le test de Johansen (ce qui correspond à `k_ar_diff = 1` et `k_ar_diff = 2` dans la représentation VECM).

**Remarque technique** : la sélection d’ordre est désormais réalisée entièrement avec `makystats.select_var_order` (échantillon commun). Les ordres retenus (AIC=3, HQ=2, BIC=1) sont **identiques** à ceux de l’article JSS.

## 2. Détermination du rang de cointégration (Johansen)

### Spécification déterministe

Nous utilisons le **cas 4** de Johansen (équivalent `ecdet = "trend"` dans `urca::ca.jo`) :
- constante **non restreinte** dans le VECM (permet une dérive linéaire dans les niveaux),
- **tendance linéaire restreinte** dans la relation de cointégration.

Ce choix est motivé par la présence évidente de tendances dans les séries (emploi, productivité, salaires) et par la volonté de permettre une tendance déterministe dans la relation de long terme (ex. : progrès technique, institutions du marché du travail).

In [4]:
print("=== Résumé complet des 5 cas déterministes (makystats) ===\n")
for p in (3, 2):
    print(f"\n{'='*70}")
    print(f"p = {p}  (k_ar_diff = {p-1})")
    print('='*70)
    summarize_johansen(can, k_ar_diff=p-1, signif=0.05)
    r = select_coint_rank(can, k_ar_diff=p-1, test_case="4")
    print(r.summary())

=== Résumé complet des 5 cas déterministes (makystats) ===


p = 3  (k_ar_diff = 2)
                   Johansen — Summarize all deterministics                    
Series: Y1 Y2 Y3 Y4    Lags interval: 1 to 2    Included obs: 81    Rank selection: 5%
------------------------------------------------------------------------------
Data Trend:              None         None       Linear       Linear    Quadratic
                 No Intercept    Intercept    Intercept    Intercept    Intercept
                     No Trend     No Trend     No Trend        Trend        Trend
Rank or No.            Case 1       Case 2       Case 3       Case 4       Case 5
------------------------------------------------------------------------------
Trace test:
None                  89.0654     100.9408      70.9576      84.9170      72.0770
At most 1             24.1478      34.0893      27.1400      36.4184      33.0432
At most 2             10.7440      15.3244      10.7819      18.7197      15.4918
At mos

### Résultats du test de la trace (case 4) – comparaison exacte avec JSS

| Hypothèse | p=3 (k=2) | p=2 (k=1) | Article JSS (p=3 / p=2) |
|-----------|-----------|-----------|-------------------------|
| r = 0     | 84.92     | 86.12     | 84.92 / 86.12           |
| r ≤ 1     | 36.42     | 37.33     | 36.42 / 37.33           |
| r ≤ 2     | 18.72     | 15.65     | 18.72 / 15.65           |
| r ≤ 3     |  3.85     |  4.10     |  3.85 /  4.10           |

**Conclusion statistique** : on rejette r = 0 au seuil de 5 % (et même 1 %), mais on ne rejette pas r ≤ 1. Le rang de cointégration retenu est donc **r = 1** pour les deux ordres de retards.

Les valeurs critiques utilisées par `makystats` (MacKinnon–Haug–Michelis 1999) sont légèrement différentes de celles d’Osterwald-Lenum (1992) utilisées dans certaines versions de `urca`, mais les statistiques de test elles-mêmes sont **identiques** à celles de l’article.

## 3. Estimation du VECM (r = 1, k_ar_diff = 2, case 4b)

Nous estimons le modèle avec l’ordre p = 3 (k_ar_diff = 2) et le rang r = 1, en utilisant la spécification déterministe `4b` de `makystats` (constante libre + tendance restreinte dans la cointégration).  
L’ordre des variables est volontairement `rw, prod, e, U` pour obtenir la normalisation sur le salaire réel, comme dans le Tableau 5 de l’article JSS.

In [5]:
# Ordre exact de l'article pour la normalisation
data_ord = can[["rw", "prod", "e", "U"]]

m = VECM(data_ord, k_ar_diff=2, coint_rank=1, det_case="4b").fit()
print(m.summary())

print("\n" + "="*60)
print("Vecteur de cointégration β (normalisé sur rw)")
print("="*60)
print(m.beta)

print("\n" + "="*60)
print("Coefficients d'ajustement α")
print("="*60)
print(m.alpha)

                        Vector Error Correction Model                         
Endogenous variables:   rw prod e U                    No. Observations: 81
Deterministic case:     4b                            Cointegrating rank: 1

Cointegrating Eq: CointEq1
                      coef     std err         z     P>|z|
------------------------------------------------------------------------------
rw(-1)              1.0000           .         .         .   (normalisé à 1)
prod(-1)            0.5449      0.6463     0.843     0.399
e(-1)              -0.0130      0.7240    -0.018     0.986
U(-1)               1.7266      1.5455     1.117     0.264
@TREND             -0.7092      0.2948    -2.406     0.016
------------------------------------------------------------------------------

Equation: D(rw)
                      coef     std err         t     P>|t|
------------------------------------------------------------------------------
COINTEQ1           -0.0848      0.0159    -5.348     0.0

### Comparaison exacte avec le Tableau 5 de l’article JSS

**Vecteur de cointégration β̂′** (normalisé sur rw) :

|          | prod   | e      | U      | rw   | trend  |
|----------|--------|--------|--------|------|--------|
| **makystats** | 0.545  | −0.013 | 1.727  | 1    | −0.709 |
| **Article JSS** | 0.545  | −0.013 | 1.727  | 1    | −0.709 |

**Coefficients d’ajustement α̂′** :

|          | prod    | e       | U       | rw      |
|----------|---------|---------|---------|---------|
| **makystats** | −0.012  | −0.016  | −0.009  | −0.085  |
| **Article JSS** | −0.012  | −0.016  | −0.009  | −0.085  |

**Correspondance numérique parfaite** (aux arrondis d’affichage près).

### Interprétation économique et économétrique approfondie

#### 3.1 La relation de long terme (β)

La relation de cointégration normalisée s’écrit approximativement :

$$
rw_t = -0.545\, prod_t + 0.013\, e_t - 1.727\, U_t + 0.709\, t + \varepsilon_t
$$

(en ignorant le signe de la normalisation pour l’interprétation économique).

Plus formellement, la combinaison stationnaire est :

$$
\beta' y_t + \rho t = rw_t + 0.545\, prod_t - 0.013\, e_t + 1.727\, U_t - 0.709\, t \sim I(0)
$$

**Lecture économique** :  
Il s’agit d’une **relation de fixation des salaires** (wage-setting relation) augmentée d’une tendance.  
- Le coefficient positif de `prod` indique que les gains de productivité se traduisent, à long terme, par une hausse des salaires réels (élasticité ≈ 0,55).  
- Le coefficient positif et important de `U` signifie qu’une hausse du chômage exerce une pression à la baisse sur les salaires réels (mécanisme de négociation / courbe de Phillips de long terme).  
- Le coefficient de l’emploi (`e`) est quasi-nul et non significatif → l’emploi n’entre pas de façon indépendante dans la relation de salaire une fois la productivité et le chômage contrôlés.  
- La tendance négative (−0,71) capture une dérive déterministe (évolution des institutions, composition de la main-d’œuvre, etc.).

Cette interprétation est celle retenue dans la littérature (Breitung et al., 2004 ; Pfaff, 2008) : la relation de cointégration est une relation de **wage-setting** stationnaire autour d’une tendance.

#### 3.2 Les coefficients d’ajustement (α)

| Variable | α̂     | \|t\| approx. | Interprétation |
|----------|--------|------------|----------------|
| rw       | −0.085 | ≈ 5.3      | Ajustement **fort et significatif** : les salaires réels corrigent rapidement les déséquilibres de long terme. |
| e        | −0.016 | ≈ 2.0      | Ajustement modéré de l’emploi. |
| prod     | −0.012 | ≈ 0.9      | Quasi-exogène faible (non significatif). |
| U        | −0.009 | ≈ 1.4      | Ajustement lent / faible du chômage. |

**Implication** : le salaire réel est la variable qui assure l’essentiel de la correction d’erreur. La productivité apparaît faiblement exogène, ce qui est cohérent avec l’idée que les chocs de productivité sont largement permanents et peu influencés par les déséquilibres du marché du travail à court terme.

#### 3.3 Remarques techniques importantes

- La normalisation sur `rw` est une convention d’identification ; toute rotation du système de cointégration (ici r=1) donnerait une relation économiquement équivalente.  
- Les écarts-types de β sont obtenus sous l’hypothèse de normalité asymptotique après estimation du rang. Ils doivent être interprétés avec prudence en petit échantillon (T≈80).  
- Le cas déterministe 4b autorise une tendance linéaire dans la relation de cointégration tout en permettant une dérive dans les variables en niveaux. C’est le cas le plus flexible (et le plus souvent pertinent) lorsque les séries présentent des tendances stochastiques **et** déterministes.

## 4. Exercice de prévision hors-échantillon

On laisse de côté les 8 derniers trimestres (1999Q1–2000Q4) et on ré-estime le VECM sur l’échantillon restreint, puis on compare les prévisions de `makystats` (case 4b) et de `statsmodels` (spécification `colo`).

In [9]:
train, test = can.iloc[:-8], can.iloc[-8:]
print("Train :", train.index[0], "→", train.index[-1], f"({len(train)} obs)")
print("Test  :", test.index[0], "→", test.index[-1], f"({len(test)} obs)")

# makystats
mk = VECM(train, k_ar_diff=2, coint_rank=1, det_case="4b").fit()
fc_mk = mk.forecast(steps=8, conf_int=True, method="analytic")


def extract_point(fc):
    if isinstance(fc, tuple):
        fc = fc[0]
    return np.asarray(getattr(fc, "values", fc), dtype=float)[:, :4]

a_mk = extract_point(fc_mk)

print("\n--- Prévisions makystats (point + IC) ---")
print(fc_mk)
print("\n--- Valeurs réalisées ---")
print(test)

Train : 1980Q1 → 1998Q4 (76 obs)
Test  : 1999Q1 → 2000Q4 (8 obs)

--- Prévisions makystats (point + IC) ---
            e      prod        rw       U   e_lower   e_upper  prod_lower  prod_upper  rw_lower  rw_upper  U_lower  \
T+1  957.7701  413.3228  467.5927  7.5153  957.1287  958.4115    412.1048    414.5407  466.1765  469.0088   6.9670   
T+2  958.7261  413.8490  468.6268  7.1406  957.3646  960.0876    411.8836    415.8143  466.5094  470.7442   6.1532   
T+3  959.4789  413.9240  469.6005  6.9905  957.3232  961.6346    411.3215    416.5264  466.9729  472.2280   5.4735   
T+4  960.0305  413.8251  470.7619  6.8745  957.0882  962.9728    410.7164    416.9338  467.6104  473.9134   4.8031   
T+5  960.4158  413.8218  472.0976  6.8532  956.7564  964.0753    410.2795    417.3641  468.4952  475.7000   4.2820   
T+6  960.7304  413.8781  473.4867  6.9237  956.4422  965.0185    409.9406    417.8155  469.5625  477.4109   3.9222   
T+7  961.0286  413.9201  474.7667  7.0074  956.2006  965.8565    4

## 5. Synthèse et points de vigilance

### Ce qui a été parfaitement reproduit
1. Sélection d’ordre VAR via `makystats.select_var_order` (AIC/HQ/BIC) → **identique** à JSS.  
2. Statistiques de la trace du test de Johansen (case 4) pour p=2 et p=3 → **identiques**.  
3. Vecteur de cointégration β et coefficients d’ajustement α du VECM (r=1, K=3) → **identiques** au Tableau 5 de l’article.

### Points de vigilance méthodologiques
- **Choix du cas déterministe** : le cas 4 (tendance restreinte) est le plus pertinent ici, mais il doit être justifié par des tests de restriction (ou par la théorie économique). Un mauvais choix de cas peut biaiser le rang et les coefficients.  
- **Petit échantillon** (T=84, T effectif ≈ 80) : les distributions asymptotiques des tests de rang et des t-stats de β sont approximatives. Des bootstrap ou des corrections de Bartlett peuvent être utiles.  
- **Normalisation** : la normalisation sur `rw` est une pure convention d’identification. Toute interprétation économique doit rester invariante à une rotation du système.  
- **Exogénéité faible** : la productivité apparaît faiblement exogène (α non significatif). Cela justifie éventuellement des analyses conditionnelles ou des SVEC avec restrictions de long terme supplémentaires (comme dans la suite de l’article JSS).  


### Extensions naturelles (non réalisées ici)
- Tests de restrictions sur β (ex. : homogénéité de long terme salaire/productivité).  
- Analyse SVEC avec identification des chocs (comme dans la section SVEC de l’article JSS).  
- Diagnostic des résidus (normalité, autocorrélation, ARCH, stabilité des paramètres) via `res.plot_diagnostics()` / `test_whiteness()` / `test_normality()`.  

---

**Conclusion** : l’implémentation `makystats` (≥ 0.1.1) reproduit fidèlement les résultats de référence de Pfaff (2008) sur le jeu de données Canada, y compris la sélection d’ordre VAR native. L’analyse confirme l’existence d’une unique relation de cointégration interprétable comme une relation de fixation des salaires, dans laquelle le salaire réel assure l’essentiel de la correction d’erreur.

## Annexe : session info et reproductibilité

In [10]:
import statsmodels, makystats, sys
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("statsmodels :", statsmodels.__version__)
print("makystats   :", getattr(makystats, "__version__", "non disponible"))

Python      : 3.14.6
NumPy       : 2.5.2
Pandas      : 3.0.5
statsmodels : 0.15.0
makystats   : 0.1.6
